# Train UniDepthLSS

Train binary vehicle segmentation from the six NuScenes cameras.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from dataset_nuscenes import NuScenesBEVDataset
from model import UniDepthLSS
from train_utils import run_epoch

DATA_ROOT = Path(os.environ.get("NUSCENES_ROOT", "/path/to/nuscenes"))
CHECKPOINT_DIR = Path(os.environ.get("UNIDEPTHLSS_CHECKPOINT_DIR", "checkpoints"))
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

VERSION = "v1.0-trainval"
IMAGE_SIZE = (294, 518)
BEV_SIZE = 128
BEV_RESOLUTION = 0.5
FEATURE_CHANNELS = 128
BATCH_SIZE = 2
NUM_WORKERS = 4
EPOCHS = 30
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Set NUSCENES_ROOT to the NuScenes directory: {DATA_ROOT}")

In [ ]:
train_dataset = NuScenesBEVDataset(
    dataroot=str(DATA_ROOT), version=VERSION, split="train",
    img_size=IMAGE_SIZE, bev_size=BEV_SIZE, bev_res=BEV_RESOLUTION,
)
val_dataset = NuScenesBEVDataset(
    dataroot=str(DATA_ROOT), version=VERSION, split="val",
    img_size=IMAGE_SIZE, bev_size=BEV_SIZE, bev_res=BEV_RESOLUTION,
    augment=False,
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
)
len(train_dataset), len(val_dataset)

In [ ]:
model = UniDepthLSS(
    img_height=IMAGE_SIZE[0], img_width=IMAGE_SIZE[1],
    num_classes=1, feature_channels=FEATURE_CHANNELS,
).to(DEVICE)
model.initialize_head_bias()

optimizer = torch.optim.AdamW(
    (parameter for parameter in model.parameters() if parameter.requires_grad),
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6,
)
scaler = torch.cuda.amp.GradScaler() if USE_AMP and DEVICE.type == "cuda" else None

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_iou": [], "val_iou": []}
best_iou = -1.0

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(
        model, train_loader, DEVICE, optimizer=optimizer, scaler=scaler,
    )
    val_metrics = run_epoch(model, val_loader, DEVICE)
    scheduler.step()

    for split, metrics in (("train", train_metrics), ("val", val_metrics)):
        history[f"{split}_loss"].append(metrics["loss"])
        history[f"{split}_iou"].append(metrics["iou"])

    print(
        f"{epoch:02d}/{EPOCHS}  "
        f"train loss {train_metrics['loss']:.4f}, IoU {train_metrics['iou']:.4f}  "
        f"val loss {val_metrics['loss']:.4f}, IoU {val_metrics['iou']:.4f}"
    )

    if val_metrics["iou"] > best_iou:
        best_iou = val_metrics["iou"]
        torch.save(
            {"epoch": epoch, "model_state_dict": model.state_dict(),
             "val_iou": best_iou, "history": history},
            CHECKPOINT_DIR / "best_model.pt",
        )

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["val_loss"], label="validation")
axes[0].set(title="Loss", xlabel="Epoch")
axes[1].plot(epochs, history["train_iou"], label="train")
axes[1].plot(epochs, history["val_iou"], label="validation")
axes[1].set(title="IoU", xlabel="Epoch")
for axis in axes:
    axis.legend()
    axis.grid(alpha=0.3)
plt.tight_layout()